# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library, following the Croissant schema FAIR^2 standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport json
# Define the dataset URL (Croissant schema)croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset objectdataset = mlc.Dataset(croissant_url)

# Access the dataset metadata (not subscriptable, so use attributes)metadata = dataset.metadata
# Print some basic infoprint(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all Record Sets with their @id and namesprint("Available Record Sets (by @id and name):")for record_set in dataset.record_sets:    print(f"- @id: {record_set['@id']}, name: {record_set['name']}")
# For each record set, list its fields and columns by their @idprint("\nFields and columns for each Record Set:")for record_set in dataset.record_sets:    print(f"\nRecord Set: {record_set['name']} (@id: {record_set['@id']})")    fields = record_set.get('field', [])    if isinstance(fields, dict):        fields = [fields]    for field in fields:        # Some field objects may be just @id references, so resolve if necessary:        # mlcroissant exposes a dictionary of all entities in dataset.entities        field_obj = dataset.entities.get(field['@id'], field)        print(f"  - Field: {field_obj.get('name', '')} (@id: {field_obj.get('@id','')})")        columns = field_obj.get('column', [])        if isinstance(columns, dict):            columns = [columns]        for col in columns:            col_obj = dataset.entities.get(col['@id'], col)            print(f"      - Column: {col_obj.get('name', '')} (@id: {col_obj.get('@id','')})")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the Record Set and Field `@id`s from the overview.

In [ ]:
# Collect all available record set @idsrecord_set_ids = [r['@id'] for r in dataset.record_sets]
# Extract all dataframes, store them in a dict keyed by their @iddataframes = {}
for record_set_id in record_set_ids:    print(f"Loading records for Record Set {record_set_id}...")    records = list(dataset.records(record_set=record_set_id))    if records:        df = pd.DataFrame(records)        dataframes[record_set_id] = df        print(f"  Loaded shape: {df.shape}")    else:        print("  No records returned or record set is empty.")
# Show the first one with datafor rsid in dataframes:    print(f"\nExample columns for Record Set {rsid}:")    print(dataframes[rsid].columns.tolist())    display(dataframes[rsid].head())    break  # Show just the first non-empty one

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a record set with tabular data (first non-empty one)
assert len(dataframes) > 0, 'No tabular data found in this dataset.'rsid = list(dataframes.keys())[0]  # Use the first record set with datadf = dataframes[rsid]print(f"Working with Record Set: {rsid}")
# Identify numeric columnsnumeric_cols = df.select_dtypes(include='number').columns.tolist()print(f"Numeric fields (@id): {numeric_cols}")
# Pick the first numeric field @id if availableif numeric_cols:    numeric_field_id = numeric_cols[0]    print(f"Selected numeric field: {numeric_field_id}")else:    print("No numeric field detected. EDA steps will be limited.")
if numeric_cols:    # Filtering records above a threshold    threshold = df[numeric_field_id].mean()  # Use mean as a simple threshold    filtered_df = df[df[numeric_field_id] > threshold]    print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")    display(filtered_df.head())
    # Normalize the numeric field (z-score)    filtered_df[f"{numeric_field_id}_normalized"] = (        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-10)    print(f"Normalized {numeric_field_id} for filtered records:")    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Find a possible group/categorical field (non-numeric but with low cardinality)    group_field = None    for c in df.columns:        if (df[c].dtype == 'object' and df[c].nunique() < 15 and c != numeric_field_id):            group_field = c            break    if group_field:        print(f"Grouped data by {group_field} (mean values):")        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()        display(grouped_df.head())    else:        print("No suitable categorical group field found for grouping.")else:    print("Skipping EDA — no numeric fields found in this Record Set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snssns.set(style='whitegrid')
if numeric_cols:    plt.figure(figsize=(8, 5))    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='navy')    plt.title(f'Distribution of {numeric_field_id}')    plt.xlabel(numeric_field_id)    plt.ylabel('Count')    plt.tight_layout()    plt.show()
    if group_field:        plt.figure(figsize=(10, 5))        sns.boxplot(x=group_field, y=numeric_field_id, data=df)        plt.title(f'{numeric_field_id} grouped by {group_field}')        plt.xticks(rotation=45)        plt.tight_layout()        plt.show()else:    print("No numeric columns available for plotting.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 Croissant dataset describing mass spectrometry analysis on extracellular vesicles from human mast cells. We demonstrated how to enumerate record sets and fields by their `@id`, extract tabular data, filter and normalize numeric fields, group by categorical fields, and visualize results.

Further analysis can include more advanced statistical or domain-specific downstream tasks, enabled by the semantic structure exposed via the Croissant schema and `mlcroissant` tools.